# PQI Pipeline , Pressure Quality Index

Computes the **Pressure Quality Index (PQI)** for every player × phase across all 5 Bundesliga matches.

PQI measures how well a player executes pressing actions, combining three sub-scores:
- **Orientation** , body orientation relative to the ball carrier
- **Stance** , body posture readiness during the press
- **Proximity** , distance to the ball carrier at press time

**Output:** `results/pqi_full.csv` , 400 rows (20 players × 2 halves × 5 matches × 2 teams).

**Checkpoint:** completed phases are saved incrementally; re-running skips already-processed phases.

## Step 1. Setup: AWS Session and Configuration

Initialises the AWS session and S3 client needed to stream skeleton Parquet files.
Sets the S3 bucket, challenge prefix, and checkpoint path.

In [ ]:
import os
import sys
sys.path.insert(0, "..")

from src.eda_helpers import create_session
from src.pressure_pipeline import run_all_matches_pqi, MATCH_CONFIGS

session, s3_client, s3fs = create_session()
BUCKET = os.environ.get("HACKATHON_BUCKET", "your-s3-bucket-name")
PREFIX = "Challenge 2 – Unlock the Power of 3D Football Data/Match_Data"
CHECKPOINT = "../results/pqi_checkpoint.csv"

## Step 2. Run PQI Pipeline

Streams each match's skeleton Parquet from S3 and computes PQI sub-scores per player per phase.

Progress is checkpointed after each phase so the pipeline can be safely interrupted and resumed.

In [ ]:
pqi_df = run_all_matches_pqi(
    s3_client,
    s3fs,
    BUCKET,
    PREFIX,
    match_configs=MATCH_CONFIGS,
    checkpoint_path=CHECKPOINT,
)

## Step 3. Save Results

Writes the full PQI DataFrame to `results/pqi_full.csv` and prints a shape/preview summary.

In [ ]:
pqi_df.to_csv("../results/pqi_full.csv", index=False)
print(f"Shape: {pqi_df.shape}")
print(pqi_df.head())

## Step 4. Validation

Asserts that the output has the expected 400 rows and all required columns are present.
Fails loudly if the pipeline produced incomplete or malformed output.

In [ ]:
assert len(pqi_df) == 400, f"Expected 400 rows, got {len(pqi_df)}"
required_cols = [
    "jersey", "team", "name", "position", "match_id", "phase_label",
    "phase_start", "phase_end", "mean_pqi", "median_pqi", "std_pqi",
    "n_press_frames", "press_minutes", "orientation_mean", "stance_mean",
    "proximity_mean", "coverage_pct"
]
missing = [c for c in required_cols if c not in pqi_df.columns]
assert not missing, f"Missing columns: {missing}"
print("\u2713 All validations passed")